# Conformer Speech Enhancement — Training Notebook

This notebook trains a **Conformer-based speech enhancement model** end-to-end on **Google Colab (free tier)**.

**What this notebook does:**
1. Installs dependencies
2. Clones/mounts the project code
3. Downloads the **VoiceBank+DEMAND** dataset (noisy/clean speech pairs) automatically
4. Builds the Conformer model (medium config, tuned for a free Colab T4 GPU)
5. Trains with mixed precision, checkpointing, and resumability (handles Colab session timeouts)
6. Evaluates with PESQ / STOI / SI-SDR
7. Exports the final checkpoint for the Streamlit app

> ⚠️ **Before running:** In Colab, go to `Runtime > Change runtime type > T4 GPU` (free tier).

---


## 1. Setup & Dependencies

In [ ]:
!pip install -q torch torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install -q soundfile librosa pesq pystoi tqdm tensorboard
print("Dependencies installed.")


In [ ]:
import torch
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: No GPU detected. Go to Runtime > Change runtime type > T4 GPU.")


## 2. Get the project code

Option A (recommended): clone from your GitHub repo after you push this project there.
Option B: upload the project zip directly to Colab and unzip it (used below by default).


In [ ]:
import os

PROJECT_DIR = "/content/conformer-speech-enhancement"

# ---- Option B: Upload the project zip (default) ----
if not os.path.exists(PROJECT_DIR):
    from google.colab import files
    print("Please upload the 'conformer-speech-enhancement.zip' project file...")
    uploaded = files.upload()
    zip_name = list(uploaded.keys())[0]
    !unzip -q "{zip_name}" -d /content/
    # Handle possible nested folder name
    if not os.path.exists(PROJECT_DIR):
        extracted = [d for d in os.listdir("/content") if os.path.isdir(f"/content/{d}") and "conformer" in d.lower()]
        if extracted:
            PROJECT_DIR = f"/content/{extracted[0]}"

os.chdir(PROJECT_DIR)
print("Working directory:", os.getcwd())

# ---- Option A: Uncomment to clone from GitHub instead ----
# !git clone https://github.com/<your-username>/conformer-speech-enhancement.git {PROJECT_DIR}
# os.chdir(PROJECT_DIR)


In [ ]:
import sys
sys.path.insert(0, PROJECT_DIR)
print("Project files:")
!ls -la


## 3. Download the VoiceBank+DEMAND dataset

We use the **VoiceBank+DEMAND** corpus (Valentini-Botinhao et al.), the standard
benchmark for speech enhancement. It is ~2.5 GB and downloads directly from the
University of Edinburgh's public DataShare (no login required, fully free).

If the primary mirror is unavailable, a HuggingFace mirror is used as fallback.


In [ ]:
import os, urllib.request, zipfile, shutil
from tqdm import tqdm

DATA_DIR = os.path.join(PROJECT_DIR, "data")
os.makedirs(DATA_DIR, exist_ok=True)

# Official Edinburgh DataShare direct-download links (VoiceBank+DEMAND, 28-speaker train set)
FILES = {
    "clean_trainset_28spk_wav.zip": "https://datashare.ed.ac.uk/bitstream/handle/10283/2791/clean_trainset_28spk_wav.zip",
    "noisy_trainset_28spk_wav.zip": "https://datashare.ed.ac.uk/bitstream/handle/10283/2791/noisy_trainset_28spk_wav.zip",
    "clean_testset_wav.zip":        "https://datashare.ed.ac.uk/bitstream/handle/10283/2791/clean_testset_wav.zip",
    "noisy_testset_wav.zip":        "https://datashare.ed.ac.uk/bitstream/handle/10283/2791/noisy_testset_wav.zip",
}

class DownloadProgressBar(tqdm):
    def update_to(self, b=1, bsize=1, tsize=None):
        if tsize is not None:
            self.total = tsize
        self.update(b * bsize - self.n)

def download(url, dest):
    if os.path.exists(dest):
        print(f"Already downloaded: {dest}")
        return
    with DownloadProgressBar(unit="B", unit_scale=True, miniters=1, desc=os.path.basename(dest)) as t:
        urllib.request.urlretrieve(url, dest, reporthook=t.update_to)

for fname, url in FILES.items():
    dest = os.path.join(DATA_DIR, fname)
    try:
        download(url, dest)
    except Exception as e:
        print(f"Primary mirror failed for {fname}: {e}")
        print("Falling back to HuggingFace mirror...")
        hf_url = f"https://huggingface.co/datasets/JorisCos/VoiceBank-DEMAND-16k/resolve/main/{fname}"
        download(hf_url, dest)

print("\nAll downloads finished.")


In [ ]:
# Extract archives
for fname in FILES.keys():
    zpath = os.path.join(DATA_DIR, fname)
    extract_marker = zpath.replace(".zip", "")
    if os.path.exists(extract_marker):
        print(f"Already extracted: {extract_marker}")
        continue
    print(f"Extracting {fname} ...")
    with zipfile.ZipFile(zpath, "r") as zf:
        zf.extractall(DATA_DIR)

print("\nDataset directory structure:")
!ls {DATA_DIR}


In [ ]:
# Sanity check: counts + resample to 16kHz mono if needed
import glob, soundfile as sf

for split_dir in ["clean_trainset_28spk_wav", "noisy_trainset_28spk_wav", "clean_testset_wav", "noisy_testset_wav"]:
    full = os.path.join(DATA_DIR, split_dir)
    n = len(glob.glob(os.path.join(full, "*.wav")))
    print(f"{split_dir}: {n} files")

# Peek at native sample rate (VoiceBank+DEMAND ships at 48kHz; our pipeline resamples
# on-the-fly to 16kHz inside the Dataset class, so no bulk conversion needed here).
sample_file = glob.glob(os.path.join(DATA_DIR, "clean_trainset_28spk_wav", "*.wav"))[0]
info = sf.info(sample_file)
print(f"\nSample file: {sample_file}")
print(f"Native sample rate: {info.samplerate} Hz (auto-resampled to 16kHz during loading)")


## 4. Build the Conformer model

Medium configuration, sized to comfortably fit a free-tier Colab T4 (16 GB VRAM)
with mixed-precision training.


In [ ]:
from model.conformer import build_model
from model.losses import ConformerSELoss
from model.dataset import VoiceBankDemandDataset, collate_fn
from utils.audio import stft, istft, complex_to_log_mag, apply_crm, SAMPLE_RATE

CONFIG = {
    "n_fft": 512,
    "d_model": 256,        # medium width
    "num_layers": 8,       # medium depth
    "num_heads": 4,
    "conv_kernel": 31,
    "ff_expansion": 4,
    "dropout": 0.1,
    "mask_bound": 3.0,

    # Training hyperparameters
    "batch_size": 8,
    "segment_seconds": 4.0,
    "lr": 3e-4,
    "weight_decay": 1e-6,
    "epochs": 60,
    "warmup_steps": 1000,
    "grad_clip": 5.0,
    "sdr_weight": 1.0,
    "spec_weight": 1.0,
    "checkpoint_every_steps": 500,
    "num_workers": 2,
}

device = "cuda" if torch.cuda.is_available() else "cpu"
model = build_model(CONFIG).to(device)
print(f"Model parameters: {model.count_parameters():,}")


## 5. Data loaders

In [ ]:
from torch.utils.data import DataLoader

train_dataset = VoiceBankDemandDataset(
    root_dir=DATA_DIR, split="train",
    segment_seconds=CONFIG["segment_seconds"], random_crop=True,
)
val_dataset = VoiceBankDemandDataset(
    root_dir=DATA_DIR, split="test",
    segment_seconds=CONFIG["segment_seconds"], random_crop=False,
)

train_loader = DataLoader(
    train_dataset, batch_size=CONFIG["batch_size"], shuffle=True,
    num_workers=CONFIG["num_workers"], collate_fn=collate_fn, drop_last=True, pin_memory=True,
)
val_loader = DataLoader(
    val_dataset, batch_size=CONFIG["batch_size"], shuffle=False,
    num_workers=CONFIG["num_workers"], collate_fn=collate_fn, drop_last=False, pin_memory=True,
)

print(f"Train samples: {len(train_dataset)} | Val samples: {len(val_dataset)}")
print(f"Train batches/epoch: {len(train_loader)}")


## 6. Training loop

Includes:
- Mixed precision (AMP) for speed on the free T4
- Gradient clipping
- Cosine LR schedule with linear warmup
- Checkpointing to Google Drive (survives Colab disconnects) — **mount Drive below**
- Resume-from-checkpoint support


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

CHECKPOINT_DIR = "/content/drive/MyDrive/conformer_se_checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print("Checkpoints will be saved to:", CHECKPOINT_DIR)


In [ ]:
import math

def lr_lambda(step, warmup_steps=CONFIG["warmup_steps"], total_steps=CONFIG["epochs"] * len(train_loader)):
    if step < warmup_steps:
        return step / max(1, warmup_steps)
    progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
    return 0.5 * (1 + math.cos(math.pi * min(progress, 1.0)))

optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG["lr"], weight_decay=CONFIG["weight_decay"])
scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
scaler = torch.cuda.amp.GradScaler(enabled=(device == "cuda"))
criterion = ConformerSELoss(sdr_weight=CONFIG["sdr_weight"], spec_weight=CONFIG["spec_weight"])

start_epoch = 0
global_step = 0
best_val_loss = float("inf")

# ---- Resume from last checkpoint if present ----
last_ckpt_path = os.path.join(CHECKPOINT_DIR, "conformer_se_last.pt")
if os.path.exists(last_ckpt_path):
    print("Resuming from checkpoint:", last_ckpt_path)
    ckpt = torch.load(last_ckpt_path, map_location=device)
    model.load_state_dict(ckpt["model_state_dict"])
    optimizer.load_state_dict(ckpt["optimizer_state_dict"])
    scheduler.load_state_dict(ckpt["scheduler_state_dict"])
    start_epoch = ckpt["epoch"] + 1
    global_step = ckpt["global_step"]
    best_val_loss = ckpt.get("best_val_loss", float("inf"))
    print(f"Resumed at epoch {start_epoch}, step {global_step}")
else:
    print("No checkpoint found, starting from scratch.")


In [ ]:
def compute_batch_loss(batch, model, criterion, device):
    clean = batch["clean"].to(device, non_blocking=True)
    noisy = batch["noisy"].to(device, non_blocking=True)

    noisy_spec = stft(noisy)
    clean_spec = stft(clean)
    log_mag = complex_to_log_mag(noisy_spec)

    mask_real, mask_imag = model(log_mag)
    enh_spec = apply_crm(noisy_spec, mask_real, mask_imag)
    enh_audio = istft(enh_spec, length=noisy.shape[-1])

    loss, components = criterion(enh_audio, clean, enh_spec=enh_spec, clean_spec=clean_spec)
    return loss, components


In [ ]:
from tqdm.auto import tqdm
import time

def save_checkpoint(path, epoch, global_step, best_val_loss):
    torch.save({
        "epoch": epoch,
        "global_step": global_step,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "best_val_loss": best_val_loss,
        "config": CONFIG,
    }, path)

@torch.no_grad()
def validate():
    model.eval()
    total_loss = 0.0
    n_batches = 0
    for batch in val_loader:
        loss, _ = compute_batch_loss(batch, model, criterion, device)
        total_loss += loss.item()
        n_batches += 1
    model.train()
    return total_loss / max(1, n_batches)


print("Starting training...")
model.train()
for epoch in range(start_epoch, CONFIG["epochs"]):
    epoch_start = time.time()
    running_loss = 0.0
    total_epochs = CONFIG["epochs"]
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{total_epochs}")

    for batch in pbar:
        optimizer.zero_grad(set_to_none=True)

        with torch.cuda.amp.autocast(enabled=(device == "cuda")):
            loss, components = compute_batch_loss(batch, model, criterion, device)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), CONFIG["grad_clip"])
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        global_step += 1
        running_loss += loss.item()
        pbar.set_postfix({"loss": f"{loss.item():.3f}", "lr": f"{scheduler.get_last_lr()[0]:.2e}"})

        if global_step % CONFIG["checkpoint_every_steps"] == 0:
            save_checkpoint(last_ckpt_path, epoch, global_step, best_val_loss)

    val_loss = validate()
    epoch_time = time.time() - epoch_start
    print(f"Epoch {epoch+1} done in {epoch_time/60:.1f} min | "
          f"train_loss={running_loss/len(train_loader):.4f} | val_loss={val_loss:.4f}")

    save_checkpoint(last_ckpt_path, epoch, global_step, best_val_loss)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_ckpt_path = os.path.join(CHECKPOINT_DIR, "conformer_se_best.pt")
        save_checkpoint(best_ckpt_path, epoch, global_step, best_val_loss)
        print(f"  -> New best model saved (val_loss={val_loss:.4f})")

print("Training complete.")


## 7. Evaluation — PESQ / STOI / SI-SDR

Standard objective speech-quality metrics on the held-out test set.


In [ ]:
from pesq import pesq
from pystoi import stoi
import numpy as np

@torch.no_grad()
def evaluate_metrics(model, dataset, n_samples=50):
    model.eval()
    pesq_scores, stoi_scores, sisdr_scores = [], [], []

    indices = np.random.choice(len(dataset), size=min(n_samples, len(dataset)), replace=False)
    for idx in tqdm(indices, desc="Evaluating"):
        sample = dataset[idx]
        clean = sample["clean"].numpy()
        noisy = sample["noisy"].unsqueeze(0).to(device)

        noisy_spec = stft(noisy)
        log_mag = complex_to_log_mag(noisy_spec)
        mask_real, mask_imag = model(log_mag)
        enh_spec = apply_crm(noisy_spec, mask_real, mask_imag)
        enhanced = istft(enh_spec, length=noisy.shape[-1]).squeeze(0).cpu().numpy()

        try:
            p = pesq(SAMPLE_RATE, clean, enhanced, "wb")
            pesq_scores.append(p)
        except Exception:
            pass

        s = stoi(clean, enhanced, SAMPLE_RATE, extended=False)
        stoi_scores.append(s)

        from model.losses import si_sdr_loss
        sdr = -si_sdr_loss(torch.from_numpy(enhanced).unsqueeze(0), torch.from_numpy(clean).unsqueeze(0)).item()
        sisdr_scores.append(sdr)

    print(f"PESQ (wb):  {np.mean(pesq_scores):.3f} +/- {np.std(pesq_scores):.3f}")
    print(f"STOI:       {np.mean(stoi_scores):.3f} +/- {np.std(stoi_scores):.3f}")
    print(f"SI-SDR (dB):{np.mean(sisdr_scores):.2f} +/- {np.std(sisdr_scores):.2f}")
    return {"pesq": np.mean(pesq_scores), "stoi": np.mean(stoi_scores), "si_sdr": np.mean(sisdr_scores)}

# Load best checkpoint before evaluating
best_ckpt_path = os.path.join(CHECKPOINT_DIR, "conformer_se_best.pt")
ckpt = torch.load(best_ckpt_path, map_location=device)
model.load_state_dict(ckpt["model_state_dict"])
metrics = evaluate_metrics(model, val_dataset, n_samples=50)


## 8. Export checkpoint for the Streamlit app

Copy `conformer_se_best.pt` into the project's `checkpoints/` folder,
then include it when you deploy to Streamlit Cloud (or load it from
Google Drive / HuggingFace Hub at app startup for smaller repo size).


In [ ]:
import shutil

export_dir = os.path.join(PROJECT_DIR, "checkpoints")
os.makedirs(export_dir, exist_ok=True)
export_path = os.path.join(export_dir, "conformer_se_best.pt")
shutil.copy(best_ckpt_path, export_path)
print("Exported checkpoint to:", export_path)

# Re-zip the project with the trained checkpoint included, ready for download
shutil.make_archive("/content/conformer-speech-enhancement-trained", "zip", PROJECT_DIR)
print("Packaged project with trained weights: /content/conformer-speech-enhancement-trained.zip")

from google.colab import files
files.download("/content/conformer-speech-enhancement-trained.zip")


## 9. Quick listening test

Run inference on a single test file and listen to noisy vs. enhanced audio directly in the notebook.


In [ ]:
from model.inference import SpeechEnhancer
import IPython.display as ipd

enhancer = SpeechEnhancer(checkpoint_path=export_path, device=device)

sample = val_dataset[0]
noisy_np = sample["noisy"].numpy()
clean_np = sample["clean"].numpy()
enhanced_np = enhancer.enhance(noisy_np, sr=SAMPLE_RATE)

print("Noisy:")
ipd.display(ipd.Audio(noisy_np, rate=SAMPLE_RATE))
print("Enhanced:")
ipd.display(ipd.Audio(enhanced_np, rate=SAMPLE_RATE))
print("Clean (reference):")
ipd.display(ipd.Audio(clean_np, rate=SAMPLE_RATE))


---
## Notes on free-tier Colab

- **Session limits**: Free Colab sessions can disconnect after ~12h or due to inactivity. This notebook checkpoints to Google Drive every `checkpoint_every_steps` steps and **automatically resumes** — just re-run all cells after reconnecting.
- **GPU quota**: Free-tier GPU access is not guaranteed 24/7. If you hit a quota wall, wait a few hours or reduce `epochs`/`batch_size`.
- **Memory**: The medium config (d_model=256, 8 layers) uses well under the T4's 16GB VRAM at batch_size=8; increase batch size if you have headroom, or lower `segment_seconds` if you hit OOM.
- **Full training time**: ~60 epochs on VoiceBank+DEMAND (11,572 train pairs) takes roughly 6-10 hours on a T4, comfortably splittable across multiple free sessions thanks to checkpoint resuming.
